# MÓDULO 2: Clasificación Directa Señal Cruda -> Clase Química

Este notebook realiza el mapeo de clasificación end-to-end de la señal espectral Mössbauer cruda (transmisión relativa uniforme) a la clase cristaloquímica (8 clases de folder).

## Notación Tensorial y Formalismo Matemático
- **Entrada**: $X \in \mathbb{R}^{B \times 1 \times N}$ (donde $B$ es el batch size, $N$ es la longitud del espectro).
- **Salida**: $\hat{y} \in \mathbb{R}^{B \times 8}$ (logits pre-softmax sobre las 8 clases).
- **Función de Pérdida**: `FocalLoss` con $\gamma=2.0$ y pesos de clase $\alpha_t$ calculados por fold para mitigar el desbalance extremo de clases (Silicatos y Óxidos dominan el dataset):
  $$\mathcal{L}_{Focal}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

## Justificación Física y Arquitectónica
Los espectros Mössbauer constan de picos de absorción Lorentzianos cuya posición, ancho y profundidad determinan la clase mineral. Una red convolucional 1D es ideal para detectar estas características locales invariantes a traslaciones en velocidad. Se utiliza `AdaptiveAvgPool1d` como salvaguarda para cualquier longitud variable residual en el dataloader.

### Diagrama ASCII de CNN 1D Base:
```
X: (B, 1, N)
    │
    ▼
[Conv1d(1->32, k=7, p=3) + BatchNorm1d + ReLU + MaxPool1d(2)]  --> (B, 32, N/2)
    │
    ▼
[Conv1d(32->64, k=5, p=2) + BatchNorm1d + ReLU + MaxPool1d(2)] --> (B, 64, N/4)
    │
    ▼
[Conv1d(64->128, k=3, p=1) + BatchNorm1d + ReLU]               --> (B, 128, N/4)
    │
    ▼
[AdaptiveAvgPool1d(1)]                                         --> (B, 128, 1)
    │
    ▼
[Flatten]                                                      --> (B, 128)
    │
    ▼
[Linear(128->64) + ReLU + Dropout(0.3)]                        --> (B, 64)
    │
    ▼
[Linear(64->8)]                                                --> (B, 8)
```

### Diagrama ASCII de MLP Base:
```
X: (B, 1, N)
    │
    ▼
[AdaptiveAvgPool1d(256)]  --> (B, 1, 256)
    │
    ▼
[Flatten]                 --> (B, 256)
    │
    ▼
[Linear(256->128) + ReLU + Dropout(0.3)] --> (B, 128)
    │
    ▼
[Linear(128->64) + ReLU + Dropout(0.3)]  --> (B, 64)
    │
    ▼
[Linear(64->8)]                          --> (B, 8)
```

### Tabla de Trade-offs Computacionales
| Modelo | Nro. Parámetros Estimados | Complejidad de Operaciones (FLOPs) | Tiempo de Inferencia Estimado (CPU) |
|---|---|---|---|
| CNN 1D Base | ~36k | O(N * C_in * C_out * K) | ~1.5 ms |
| MLP Base | ~42k | O(D_in * D_out) | ~0.8 ms |

In [5]:
import os
import sys
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

# Asegurar que 'src' sea importable
sys.path.append(os.path.abspath('.'))
from src.focal_loss import FocalLoss, compute_alpha
from src.collate_fn import collate_fn
from src.metrics import plot_confusion_matrix, plot_roc_curves, plot_calibration_curve

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## 2. Carga y Preparación de Datos

In [6]:
df = pd.read_parquet('outputs/mossbauer_processed.parquet')
print(f"Dataset cargado. Shape: {df.shape}")

# Los vectores ya están homogeneizados - cargar directamente
X_data = np.stack(df['intensity_uniform'].values)  # (N_samples, target_len)
y_data = df['chem_label'].values                   # (N_samples,)
print(f"X_data shape: {X_data.shape}, y_data shape: {y_data.shape}")

Dataset cargado. Shape: (1361, 19)
X_data shape: (1361, 512), y_data shape: (1361,)


In [7]:
df.sample(1)

,pkey,Dana Class,folder,Sample Name,Owner/Source,Temperature (K),Intensity,Velocity (mm/s),velocity_filtered,intensity_filtered,velocity_uniform,intensity_uniform,v_range,chem_label,chem_onehot,topo_label,P_vec,bic_winner,fit_residual_rmse
1296,04042602,Hydrated Acid and Sulfates,Sulfate,Szomolnokite 560,"Dyar,Darby",25.0,"[-0.052, -0.014, -0.005, 0.023, 0.004, 0.018, ...","[-11.667, -11.621, -11.575, -11.529, -11.483, ...","[-11.667, -11.621, -11.575, -11.529, -11.483, ...","[-0.052, -0.014, -0.005, 0.023, 0.004, 0.018, ...","[-11.667, -11.620931506849315, -11.57486301369...","[-0.052, -0.013988581746941174, -0.00492192471...",23.541,2,"[0, 0, 1, 0, 0, 0, 0, 0]",0,"[-0.3999909956295249, 0.0, 0.0, 0.499999995277...",0,2.078981


## 3. Dataset de PyTorch

In [8]:
class MossbauerDataset(Dataset):
    def __init__(self, X, y):
        self.spectra = X
        self.labels = y
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        # Retorna el espectro como un tensor 1D de longitud variable
        # (el collate_fn se encargará del padding dinámico y del empaquetado)
        spectrum = torch.tensor(self.spectra[idx], dtype=torch.float32)
        label = self.labels[idx]
        return spectrum, label

## 4. Arquitecturas a Implementar

In [9]:
class CNN1DBase(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x):
        # Entrada: (B, 1, N)
        x = self.features(x)
        x = self.pool(x)
        x = x.squeeze(-1) # shape: (B, 128)
        x = self.classifier(x)
        return x

class MLPBase(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(256)
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x):
        # Entrada: (B, 1, N)
        x = self.pool(x)
        x = x.squeeze(1) # shape: (B, 256)
        x = self.classifier(x)
        return x

## 5. Entrenamiento y Validación Cruzada (5-Fold Stratified)

In [10]:
def train_and_eval(model_class, epochs=15, lr=0.001, batch_size=64):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    oof_probs = np.zeros((len(X_data), 8))
    oof_preds = np.zeros(len(X_data))
    
    fold_metrics = []
    os.makedirs('outputs/models', exist_ok=True)
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_data, y_data)):
        print(f"\n--- FOLD {fold+1}/5 ---")
        
        X_train, y_train = X_data[train_idx], y_data[train_idx]
        X_val, y_val = X_data[val_idx], y_data[val_idx]
        
        train_dataset = MossbauerDataset(X_train, y_train)
        val_dataset = MossbauerDataset(X_val, y_val)
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
        
        model = model_class(num_classes=8).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        
        # FocalLoss con pesos calculados para el conjunto de entrenamiento de este fold
        alpha = compute_alpha(y_train, num_classes=8).to(device)
        criterion = FocalLoss(alpha=alpha, gamma=2.0)
        
        best_val_loss = float('inf')
        
        for epoch in range(epochs):
            model.train()
            train_loss = 0.0
            for batch_x, batch_y, _ in train_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                optimizer.zero_grad()
                logits = model(batch_x)
                loss = criterion(logits, batch_y)
                loss.backward()
                optimizer.step()
                train_loss += loss.item() * len(batch_y)
            train_loss /= len(train_dataset)
            
            # Evaluación de validación para monitorear loss
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for batch_x, batch_y, _ in val_loader:
                    batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                    logits = model(batch_x)
                    loss = criterion(logits, batch_y)
                    val_loss += loss.item() * len(batch_y)
            val_loss /= len(val_dataset)
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.state_dict(), f'outputs/models/{model_class.__name__}_fold{fold+1}.pt')
                
        # Cargar el mejor modelo
        model.load_state_dict(torch.load(f'outputs/models/{model_class.__name__}_fold{fold+1}.pt'))
        model.eval()
        
        val_probs = []
        with torch.no_grad():
            for batch_x, batch_y, _ in val_loader:
                batch_x = batch_x.to(device)
                logits = model(batch_x)
                probs = F.softmax(logits, dim=-1)
                val_probs.append(probs.cpu().numpy())
                
        val_probs = np.concatenate(val_probs, axis=0)
        val_preds = np.argmax(val_probs, axis=1)
        
        oof_probs[val_idx] = val_probs
        oof_preds[val_idx] = val_preds
        
        acc = accuracy_score(y_val, val_preds)
        macro_f1 = f1_score(y_val, val_preds, average='macro')
        weighted_f1 = f1_score(y_val, val_preds, average='weighted')
        
        print(f"Fold {fold+1} Metrics - Accuracy: {acc:.4f}, Macro-F1: {macro_f1:.4f}, Weighted-F1: {weighted_f1:.4f}")
        fold_metrics.append([acc, macro_f1, weighted_f1])
        
    fold_metrics = np.array(fold_metrics)
    means = fold_metrics.mean(axis=0)
    stds = fold_metrics.std(axis=0)
    
    print(f"\n=== {model_class.__name__} FINAL METRICS ===")
    print(f"Accuracy:    {means[0]:.4f} ± {stds[0]:.4f}")
    print(f"Macro-F1:    {means[1]:.4f} ± {stds[1]:.4f}")
    print(f"Weighted-F1: {means[2]:.4f} ± {stds[2]:.4f}")
    
    return oof_probs, oof_preds, means, stds

## 6. Ejecución del Entrenamiento

In [ ]:
print("=== Entrenando CNN 1D Base ===")
cnn_probs, cnn_preds, cnn_means, cnn_stds = train_and_eval(CNN1DBase, epochs=200, lr=0.001)

print("\n=== Entrenando MLP Base ===")
mlp_probs, mlp_preds, mlp_means, mlp_stds = train_and_eval(MLPBase, epochs=200, lr=0.001)

=== Entrenando CNN 1D Base ===

--- FOLD 1/5 ---
Fold 1 Metrics - Accuracy: 0.1062, Macro-F1: 0.0320, Weighted-F1: 0.0204

--- FOLD 2/5 ---
Fold 2 Metrics - Accuracy: 0.0662, Macro-F1: 0.0565, Weighted-F1: 0.0306

--- FOLD 3/5 ---
Fold 3 Metrics - Accuracy: 0.1066, Macro-F1: 0.0321, Weighted-F1: 0.0205

--- FOLD 4/5 ---
Fold 4 Metrics - Accuracy: 0.0699, Macro-F1: 0.0459, Weighted-F1: 0.0433

--- FOLD 5/5 ---
Fold 5 Metrics - Accuracy: 0.5882, Macro-F1: 0.1518, Weighted-F1: 0.4494

=== CNN1DBase FINAL METRICS ===
Accuracy:    0.1874 ± 0.2011
Macro-F1:    0.0637 ± 0.0450
Weighted-F1: 0.1128 ± 0.1685

=== Entrenando MLP Base ===

--- FOLD 1/5 ---
Fold 1 Metrics - Accuracy: 0.4579, Macro-F1: 0.3823, Weighted-F1: 0.4970

--- FOLD 2/5 ---
Fold 2 Metrics - Accuracy: 0.6213, Macro-F1: 0.3751, Weighted-F1: 0.5660

--- FOLD 3/5 ---
Fold 3 Metrics - Accuracy: 0.4522, Macro-F1: 0.3077, Weighted-F1: 0.4832

--- FOLD 4/5 ---
Fold 4 Metrics - Accuracy: 0.2868, Macro-F1: 0.2063, Weighted-F1: 0.3493



## 7. Generación de Gráficos de Evaluación y Parámetros en JSON

In [12]:
CLASS_NAMES = ['Silicatos', 'Óxidos/Hidróxidos', 'Sulfatos', 'Sulfuros/Teluruos',
               'Oxisales', 'Haluros', 'Metales', 'Amorfos']

os.makedirs('outputs/results', exist_ok=True)

# Guardar gráficos para CNN
plot_confusion_matrix(y_data, cnn_preds, CLASS_NAMES, 'outputs/results/02_cnn_confusion_matrix.png')
plot_roc_curves(y_data, cnn_probs, CLASS_NAMES, 'outputs/results/02_cnn_roc_curves.png')
plot_calibration_curve(y_data, cnn_probs, 'outputs/results/02_cnn_calibration.png')

# Guardar gráficos para MLP
plot_confusion_matrix(y_data, mlp_preds, CLASS_NAMES, 'outputs/results/02_mlp_confusion_matrix.png')
plot_roc_curves(y_data, mlp_probs, CLASS_NAMES, 'outputs/results/02_mlp_roc_curves.png')
plot_calibration_curve(y_data, mlp_probs, 'outputs/results/02_mlp_calibration.png')

# Exportar resultados JSON
results_dict = {
    "cnn_accuracy_mean": float(cnn_means[0]),
    "cnn_accuracy_std": float(cnn_stds[0]),
    "cnn_macro_f1_mean": float(cnn_means[1]),
    "cnn_macro_f1_std": float(cnn_stds[1]),
    "cnn_weighted_f1_mean": float(cnn_means[2]),
    "cnn_weighted_f1_std": float(cnn_stds[2]),
    "mlp_accuracy_mean": float(mlp_means[0]),
    "mlp_accuracy_std": float(mlp_stds[0]),
    "mlp_macro_f1_mean": float(mlp_means[1]),
    "mlp_macro_f1_std": float(mlp_stds[1]),
    "mlp_weighted_f1_mean": float(mlp_means[2]),
    "mlp_weighted_f1_std": float(mlp_stds[2])
}

with open('outputs/results/02_direct_classification_metrics.json', 'w') as f:
    json.dump(results_dict, f, indent=4)

print("✓ Gráficos y archivo JSON guardados exitosamente en outputs/results/")

/home/jd/Projects/MossAI/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1364: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
/home/jd/Projects/MossAI/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1364: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
/home/jd/Projects/MossAI/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1364: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
/home/jd/Projects/MossAI/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1364: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(


✓ Gráficos y archivo JSON guardados exitosamente en outputs/results/
